# Chapter 4: Vectors and Euclidean spaces

Source orientation: printed pages 65-87, PDF pages 75-97, sections 4.1-4.8. The source is used here for coverage and terminology only; the prose, examples, code, diagrams, and checks are original.

## Chapter goal

Turn the chapter's coordinate formulas into inspectable geometry. By the end, vectors should feel like portable arrows: they can be added, scaled, averaged, tested for independence, measured by an inner product, and rotated either by a matrix or by multiplying complex numbers.


## Computational translation guide

| Book idea | Computational object | Visual inspection target | Check used here |
| --- | --- | --- | --- |
| Vector sum and scalar multiple | NumPy arrays with `u + v` and `a*u` | parallelogram closure and dilation from the origin | commutativity, associativity, distributivity residuals |
| Direction and parallel segments | one-dimensional spans; 2D determinant zero means parallel | same line through the origin or equal-slope transferred segments | determinant/rank and parallel residuals |
| Linear independence | nonzero determinant or nonzero Gram determinant | parallelogram area does not collapse | `abs(det([u v])) > 0` |
| Midpoint and centroid | vector averages | medians meet at the average | median residuals and tetrahedron centroid residuals |
| Inner product | `np.dot(u, v)` | perpendicularity, projection, length, angle | dot residuals, Gram matrix positive semidefinite checks |
| Cauchy-Schwarz and triangle inequality | inequalities from the Gram determinant | projection never exceeds the vector and a shortcut is no longer than a broken path | exact SymPy identity and sampled numeric slack |
| Rotation | `R(theta) @ x` and `exp(i theta) * z` | the same rotated point from two encodings | orthogonality, determinant, composition, and matrix/complex agreement |


In [ ]:
from __future__ import annotations
import math, sys
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import plotly.graph_objects as go
import sympy as sp
from IPython.display import display


def find_book_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if candidate.name == "The-Four-Pillars-of-Geometry":
            return candidate
        nested = candidate / "The-Four-Pillars-of-Geometry"
        if nested.exists():
            return nested
    raise RuntimeError("Could not locate The-Four-Pillars-of-Geometry")


BOOK_ROOT = find_book_root(Path.cwd())
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import artifact_path, assert_artifacts, display_artifact, ensure_artifact_root, image_stats, save_json, save_table
from utils.plotting import PALETTE
from utils.transforms import rotation_matrix

CHAPTER_KEY = "chapter-04-vectors-and-euclidean-spaces"
ARTIFACT_ROOT = ensure_artifact_root(BOOK_ROOT / "artifacts" / CHAPTER_KEY)
SOURCE_SPAN = {"printed_pages": "65-87", "pdf_pages": "75-97", "sections": "4.1-4.8"}
STALE_GENERIC_ARTIFACTS = [
    "figures/visual_spine.png",
    "figures/construction_or_model.png",
    "figures/algebraic_check.png",
    "figures/invariant_heatmap.png",
    "figures/proof_state.png",
    "html/interactive_invariant_lab.html",
    "checks/invariants.json",
    "checks/artifact_manifest.json",
]
for rel in STALE_GENERIC_ARTIFACTS:
    target = ARTIFACT_ROOT / rel
    if target.exists():
        target.unlink()

ARTIFACTS = {"figures": [], "html": [], "checks": [], "tables": []}
CHECKS = {"source_span": SOURCE_SPAN}
EPS = 1e-10
np.set_printoptions(precision=4, suppress=True)


def norm(v):
    return float(np.linalg.norm(np.asarray(v, dtype=float)))


def det2(u, v):
    u = np.asarray(u, dtype=float)
    v = np.asarray(v, dtype=float)
    return float(u[0] * v[1] - u[1] * v[0])


def save_figure(fig, filename):
    path = artifact_path(ARTIFACT_ROOT, "figures", filename)
    fig.savefig(path, dpi=170, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)
    ARTIFACTS["figures"].append(path)
    return path


def save_html_figure(fig, filename):
    path = artifact_path(ARTIFACT_ROOT, "html", filename)
    fig.write_html(path, include_plotlyjs=True, full_html=True)
    ARTIFACTS["html"].append(path)
    return path


def save_check(data, filename):
    path = save_json(data, ARTIFACT_ROOT, "checks", filename)
    ARTIFACTS["checks"].append(path)
    return path


def save_csv(rows, filename):
    path = save_table(rows, ARTIFACT_ROOT, "tables", filename)
    ARTIFACTS["tables"].append(path)
    return path


def style_plane(ax, title=None):
    ax.axhline(0, color="#9aa2a9", lw=0.8)
    ax.axvline(0, color="#9aa2a9", lw=0.8)
    ax.grid(True, color="#e8edf1", lw=0.7)
    ax.set_aspect("equal", adjustable="box")
    if title:
        ax.set_title(title, color=PALETTE["ink"], fontsize=12, weight="bold", pad=10)


def draw_vec(ax, start, vec, color="blue", label=None, lw=2.2, alpha=1.0):
    start = np.asarray(start, dtype=float)
    vec = np.asarray(vec, dtype=float)
    end = start + vec
    ax.annotate("", xy=end, xytext=start, arrowprops={"arrowstyle": "->", "lw": lw, "color": PALETTE.get(color, color), "alpha": alpha})
    if label:
        mid = start + 0.56 * vec
        ax.text(mid[0], mid[1], label, color=PALETTE.get(color, color), fontsize=10, weight="bold")
    return end


def label_point(ax, name, p, dx=0.05, dy=0.06):
    p = np.asarray(p, dtype=float)
    ax.scatter([p[0]], [p[1]], s=35, color=PALETTE["ink"], zorder=5)
    ax.text(p[0] + dx, p[1] + dy, name, color=PALETTE["ink"], fontsize=9, weight="bold")


print(f"Book root: {BOOK_ROOT.relative_to(BOOK_ROOT.parent)}")
print(f"Artifact root: {ARTIFACT_ROOT.relative_to(BOOK_ROOT)}")


## Source-specific storyboard

The chapter moves from operations that do not need length to measurements that do. The visual sequence follows that order: first the algebra of arrows, then direction and dependence, then averages, then the inner product, then inequalities, and finally rotations by two equivalent encodings. Each artifact below is named by the concept it teaches rather than by the rendering tool.


In [ ]:
storyboard = [
    {"concept": "parallelogram rule and scalar dilation", "representation": "two-panel 2D vector diagram", "library": "Matplotlib", "artifact": "figures/vector-parallelogram-dilation.png", "validation": "vector-space law residuals"},
    {"concept": "direction, linear independence, and vector Thales", "representation": "determinant area plus parallel transferred segments", "library": "Matplotlib and NumPy", "artifact": "figures/direction-linear-independence.png", "validation": "rank/determinant and parallel residuals"},
    {"concept": "parallel theorem proof dependencies", "representation": "directed proof graph", "library": "NetworkX", "artifact": "figures/parallel-theorems-proof-dependency-map.png", "validation": "DAG and dependency edges"},
    {"concept": "centroids as vector averages", "representation": "triangle medians plus 3D tetrahedron HTML", "library": "Matplotlib and Plotly", "artifact": "figures/centroid-median-concurrency.png and html/tetrahedron-centroid-lines.html", "validation": "median and vertex-to-face residuals"},
    {"concept": "inner product and perpendicularity", "representation": "orthocenter/altitude diagram", "library": "Matplotlib and NumPy", "artifact": "figures/inner-product-orthogonality-altitudes.png", "validation": "altitude dot products vanish"},
    {"concept": "Cauchy-Schwarz and triangle inequality", "representation": "projection and broken-path comparison", "library": "Matplotlib, SymPy, CSV", "artifact": "figures/gram-cauchy-triangle-inequality.png", "validation": "symbolic Gram determinant and sampled slack"},
    {"concept": "rotations by matrices and complex numbers", "representation": "static overlay plus Plotly slider", "library": "Matplotlib, Plotly, SymPy", "artifact": "figures/rotation-matrix-complex-action.png and html/rotation-composition-lab.html", "validation": "orthogonality, determinant, composition, matrix/complex agreement"},
    {"concept": "applied vector navigation lab", "representation": "survey traverse with headings, centroid, and Gram diagnostics", "library": "Matplotlib, Plotly, CSV/JSON", "artifact": "figures/applied-navigation-vector-lab.png and tables/navigation-steps.csv", "validation": "endpoint sum, centroid average, path/direct inequality, Gram PSD"},
]
visual_storyboard_path = save_check({"chapter_goal": "Vectors make Euclidean ideas portable: addition, direction, averaging, measurement, inequality, and rotation.", "source_span": SOURCE_SPAN, "items": storyboard}, "visual-storyboard.json")
visual_storyboard_path.relative_to(BOOK_ROOT)


## 1. Vector operations before measurement

A vector space knows how to add arrows and scale them before it knows how long they are. The parallelogram rule is the geometric test for the coordinate formula `u + v`: the same endpoint is reached whether we travel by `u` then `v`, by `v` then `u`, or by the diagonal of the parallelogram. Scalar multiplication keeps the line through the origin fixed while changing signed scale.


In [ ]:
u = np.array([2.35, 0.8])
v = np.array([0.75, 1.75])
w = np.array([-0.55, 1.15])
a, b = 1.4, -0.65
triangle = np.array([[0.45, 0.15], [1.2, 0.35], [0.75, 0.95]])

fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.4), facecolor="#fffdf8")
ax = axes[0]
style_plane(ax, "addition: the diagonal closes the parallelogram")
draw_vec(ax, [0, 0], u, color="blue", label="u")
draw_vec(ax, [0, 0], v, color="green", label="v")
draw_vec(ax, [0, 0], u + v, color="red", label="u + v")
draw_vec(ax, u, v, color="green", lw=1.7, alpha=0.75)
draw_vec(ax, v, u, color="blue", lw=1.7, alpha=0.75)
ax.plot([u[0], u[0] + v[0]], [u[1], u[1] + v[1]], "--", color=PALETTE["gray"], lw=1.2)
ax.plot([v[0], u[0] + v[0]], [v[1], u[1] + v[1]], "--", color=PALETTE["gray"], lw=1.2)
for name, point in {"0": [0, 0], "u": u, "v": v, "u+v": u + v}.items():
    label_point(ax, name, point)
ax.set_xlim(-0.6, 4.0)
ax.set_ylim(-0.4, 3.1)

ax = axes[1]
style_plane(ax, "scalar multiplication: same direction, signed scale")
base = np.vstack([triangle, triangle[0]])
ax.plot(base[:, 0], base[:, 1], color=PALETTE["ink"], lw=2.0, label="original")
for scalar, color in [(0.55, "green"), (1.65, "blue"), (-0.85, "orange")]:
    scaled = scalar * base
    ax.plot(scaled[:, 0], scaled[:, 1], color=PALETTE[color], lw=2.0, label=f"a = {scalar:g}")
    for q in scalar * triangle:
        ax.plot([0, q[0]], [0, q[1]], color=PALETTE[color], alpha=0.25, lw=0.9)
ax.legend(loc="upper left", frameon=True, fontsize=9)
ax.set_xlim(-1.3, 2.2)
ax.set_ylim(-1.0, 1.8)
fig.suptitle("Vector addition and scalar multiplication are geometric operations on coordinates", fontsize=14, weight="bold", color=PALETTE["ink"])
vector_ops_path = save_figure(fig, "vector-parallelogram-dilation.png")
vector_residuals = {
    "sum_commutativity": norm((u + v) - (v + u)),
    "sum_associativity": norm((u + (v + w)) - ((u + v) + w)),
    "scalar_distributivity_over_sum": norm(a * (u + v) - (a * u + a * v)),
    "scalar_distributivity_over_scalars": norm((a + b) * u - (a * u + b * u)),
    "scalar_associativity": norm(a * (b * u) - ((a * b) * u)),
    "inverse_residual": norm(u + (-u)),
}
CHECKS["vector_operations"] = vector_residuals
display_artifact(vector_ops_path, width=900)
vector_residuals


## 2. Direction, parallelism, and linear independence

A nonzero vector gives a whole line through the origin: every multiple of it has the same direction. Two planar vectors are linearly independent exactly when those two direction lines do not collapse into one. The determinant is the signed area of the parallelogram they span, so it is a numerical witness for independence.

The right panel encodes the vector version of Thales: once two rays from the origin are independent, a segment parallel to the base segment forces the same scale factor on both rays.


In [ ]:
p = np.array([1.35, 0.55])
q = np.array([0.45, 1.65])
nearly = 1.45 * p + np.array([0.08, -0.10])
s = np.array([1.05, 0.35])
t = np.array([0.35, 1.25])
scale = 2.05
v_ray = scale * s
w_ray = scale * t

fig, axes = plt.subplots(1, 2, figsize=(12.7, 5.35), facecolor="#fffdf8")
ax = axes[0]
style_plane(ax, "independence is visible parallelogram area")
draw_vec(ax, [0, 0], p, color="blue", label="p")
draw_vec(ax, [0, 0], q, color="green", label="q")
draw_vec(ax, [0, 0], nearly, color="orange", label="near multiple")
par = np.array([[0, 0], p, p + q, q, [0, 0]])
ax.fill(par[:, 0], par[:, 1], color=PALETTE["purple"], alpha=0.18)
ax.plot(par[:, 0], par[:, 1], color=PALETTE["purple"], lw=1.7)
ax.text(1.15, 1.2, f"area = det(p,q) = {det2(p, q):.2f}", color=PALETTE["purple"], fontsize=10, weight="bold")
ax.text(1.95, 1.03, "near multiples\nmake small area", color=PALETTE["orange"], fontsize=9)
ax.set_xlim(-0.45, 2.6)
ax.set_ylim(-0.25, 2.6)

ax = axes[1]
style_plane(ax, "vector Thales: parallel transfer fixes one scale")
for point, label in [(s, "s"), (t, "t"), (v_ray, "v = a s"), (w_ray, "w = a t")]:
    label_point(ax, label, point)
ax.plot([0, s[0], v_ray[0]], [0, s[1], v_ray[1]], color=PALETTE["blue"], lw=2)
ax.plot([0, t[0], w_ray[0]], [0, t[1], w_ray[1]], color=PALETTE["green"], lw=2)
ax.plot([s[0], t[0]], [s[1], t[1]], color=PALETTE["gray"], lw=2.1, label="t - s")
ax.plot([v_ray[0], w_ray[0]], [v_ray[1], w_ray[1]], color=PALETTE["red"], lw=2.1, label="w - v = a(t - s)")
ax.legend(loc="upper left", fontsize=9, frameon=True)
ax.set_xlim(-0.25, 2.55)
ax.set_ylim(-0.25, 2.8)
fig.suptitle("Direction is span; independence is nonzero area; parallelism is a zero determinant", fontsize=14, weight="bold", color=PALETTE["ink"])
direction_path = save_figure(fig, "direction-linear-independence.png")
direction_checks = {
    "det_p_q": det2(p, q),
    "rank_p_q": int(np.linalg.matrix_rank(np.column_stack([p, q]))),
    "det_p_near_multiple": det2(p, nearly),
    "thales_parallel_residual": det2(w_ray - v_ray, t - s),
    "thales_scale_v_over_s": float(np.dot(v_ray, s) / np.dot(s, s)),
    "thales_scale_w_over_t": float(np.dot(w_ray, t) / np.dot(t, t)),
}
CHECKS["direction_and_independence"] = direction_checks
display_artifact(direction_path, width=900)
direction_checks


## 3. Proof scaffold: Thales to Pappus

The chapter uses vectors to revisit parallel theorems from earlier geometry. The proof engine is small: independence lets us compare coefficients on two directions, and scalar commutativity is the quiet step that makes the parallel Pappus conclusion appear. The graph below marks which algebraic facts are actually consumed.


In [ ]:
G = nx.DiGraph()
edges = [
    ("two origin lines", "linear independence"),
    ("parallel segment", "coefficient comparison"),
    ("linear independence", "coefficient comparison"),
    ("coefficient comparison", "vector Thales"),
    ("vector Thales", "Pappus labels"),
    ("Pappus labels", "u = a b w"),
    ("Pappus labels", "t = b a r"),
    ("scalar commutativity", "ab = ba"),
    ("u = a b w", "u - t = ab(w-r)"),
    ("t = b a r", "u - t = ab(w-r)"),
    ("ab = ba", "u - t = ab(w-r)"),
    ("u - t = ab(w-r)", "final parallelism"),
]
G.add_edges_from(edges)
pos = {"two origin lines": (0, 2.2), "parallel segment": (0, 1.2), "linear independence": (1.8, 2.2), "coefficient comparison": (3.6, 1.7), "vector Thales": (5.3, 1.7), "Pappus labels": (7.0, 1.7), "u = a b w": (8.8, 2.45), "t = b a r": (8.8, 1.05), "scalar commutativity": (7.0, 0.05), "ab = ba": (8.8, 0.05), "u - t = ab(w-r)": (10.8, 1.65), "final parallelism": (12.7, 1.65)}
fig, ax = plt.subplots(figsize=(13.5, 4.4), facecolor="#fffdf8")
ax.set_title("Parallel theorem dependency scaffold", fontsize=14, weight="bold", color=PALETTE["ink"], pad=12)
node_colors = [PALETTE["light"] if n != "scalar commutativity" else "#ffe7bd" for n in G.nodes]
nx.draw_networkx_nodes(G, pos, node_color=node_colors, edgecolors=PALETTE["ink"], node_size=1500, ax=ax)
nx.draw_networkx_edges(G, pos, arrowstyle="-|>", arrowsize=13, edge_color=PALETTE["gray"], width=1.4, ax=ax)
nx.draw_networkx_labels(G, pos, font_size=8.3, font_color=PALETTE["ink"], ax=ax)
ax.axis("off")
proof_path = save_figure(fig, "parallel-theorems-proof-dependency-map.png")
proof_checks = {"is_dag": bool(nx.is_directed_acyclic_graph(G)), "node_count": G.number_of_nodes(), "edge_count": G.number_of_edges(), "has_scalar_commutativity_edge": bool(G.has_edge("scalar commutativity", "ab = ba"))}
CHECKS["parallel_theorem_dependencies"] = proof_checks
display_artifact(proof_path, width=950)
proof_checks


## 4. Midpoints and centroids as averages

The midpoint of `u` and `v` is their average. The centroid of a triangle is the average of its three vertices. This is not just a formula: the average lies two-thirds of the way from each vertex to the midpoint of the opposite side. The same idea extends to a tetrahedron, where a vertex-to-opposite-face-centroid line reaches the four-point average three-quarters of the way along.


In [ ]:
U = np.array([0.25, 0.35])
V = np.array([4.05, 0.85])
W = np.array([1.35, 3.45])
M_U, M_V, M_W = 0.5 * (V + W), 0.5 * (U + W), 0.5 * (U + V)
C_tri = (U + V + W) / 3.0
median_hits = {"from_U": U + (2.0 / 3.0) * (M_U - U), "from_V": V + (2.0 / 3.0) * (M_V - V), "from_W": W + (2.0 / 3.0) * (M_W - W)}

fig, ax = plt.subplots(figsize=(8.4, 6.2), facecolor="#fffdf8")
style_plane(ax, "triangle medians meet at the vector average")
tri = np.vstack([U, V, W, U])
ax.plot(tri[:, 0], tri[:, 1], color=PALETTE["ink"], lw=2.2)
ax.fill(tri[:, 0], tri[:, 1], color=PALETTE["blue"], alpha=0.08)
for A0, M, color in [(U, M_U, "red"), (V, M_V, "green"), (W, M_W, "purple")]:
    ax.plot([A0[0], M[0]], [A0[1], M[1]], color=PALETTE[color], lw=2.0)
    ax.scatter([M[0]], [M[1]], s=55, color=PALETTE[color], zorder=4)
ax.scatter([C_tri[0]], [C_tri[1]], s=135, color=PALETTE["gold"], edgecolor=PALETTE["ink"], zorder=5)
for label, point in {"u": U, "v": V, "w": W, "(v+w)/2": M_U, "(u+w)/2": M_V, "(u+v)/2": M_W}.items():
    label_point(ax, label, point)
ax.text(C_tri[0] + 0.08, C_tri[1] + 0.08, "(u+v+w)/3", color=PALETTE["ink"], fontsize=10, weight="bold")
ax.text(2.05, 2.3, "each colored median hits\nthe same average at 2/3", color=PALETTE["ink"], fontsize=10)
ax.set_xlim(-0.25, 4.55)
ax.set_ylim(-0.15, 3.95)
centroid_path = save_figure(fig, "centroid-median-concurrency.png")

T = np.array([0.0, 0.0, 0.0])
A3 = np.array([3.0, 0.25, 0.25])
B3 = np.array([0.65, 2.75, 0.35])
C3 = np.array([0.45, 0.45, 2.55])
verts3 = np.vstack([T, A3, B3, C3])
G3 = verts3.mean(axis=0)
face_centroids = [np.vstack([A3, B3, C3]).mean(axis=0), np.vstack([T, B3, C3]).mean(axis=0), np.vstack([T, A3, C3]).mean(axis=0), np.vstack([T, A3, B3]).mean(axis=0)]
vertex_to_face = [(T, face_centroids[0]), (A3, face_centroids[1]), (B3, face_centroids[2]), (C3, face_centroids[3])]
residuals3 = [norm(start + 0.75 * (face - start) - G3) for start, face in vertex_to_face]
fig3 = go.Figure()
fig3.add_trace(go.Mesh3d(x=verts3[:, 0], y=verts3[:, 1], z=verts3[:, 2], i=[0, 0, 0, 1], j=[1, 1, 2, 2], k=[2, 3, 3, 3], opacity=0.18, color="#1f77b4", name="tetrahedron faces"))
fig3.add_trace(go.Scatter3d(x=verts3[:, 0], y=verts3[:, 1], z=verts3[:, 2], mode="markers+text", text=["t", "u", "v", "w"], textposition="top center", marker=dict(size=5, color="#243040"), name="vertices"))
for idx, (start, face) in enumerate(vertex_to_face):
    fig3.add_trace(go.Scatter3d(x=[start[0], face[0]], y=[start[1], face[1]], z=[start[2], face[2]], mode="lines", line=dict(width=5), name=f"vertex to face centroid {idx+1}"))
fig3.add_trace(go.Scatter3d(x=[G3[0]], y=[G3[1]], z=[G3[2]], mode="markers+text", text=["(t+u+v+w)/4"], textposition="bottom center", marker=dict(size=7, color="#c7971d"), name="tetrahedron centroid"))
fig3.update_layout(title="Tetrahedron centroid: vertex-to-face-centroid lines meet at the average", scene=dict(aspectmode="data", xaxis_title="x", yaxis_title="y", zaxis_title="z"), margin=dict(l=0, r=0, b=0, t=55))
tetra_html_path = save_html_figure(fig3, "tetrahedron-centroid-lines.html")

centroid_checks = {"triangle_centroid": C_tri.tolist(), "triangle_median_residuals": {k: norm(hit - C_tri) for k, hit in median_hits.items()}, "tetrahedron_centroid": G3.tolist(), "tetrahedron_vertex_to_face_residuals": residuals3}
CHECKS["centroids"] = centroid_checks
display_artifact(centroid_path, width=760)
display_artifact(tetra_html_path, width="100%", height=520)
centroid_checks


## 5. Inner products: length, angle, and orthogonality

The inner product adds measurement to vector-space algebra. It gives squared length by `u dot u`, tests perpendicularity by `u dot v = 0`, and later gives the cosine of the angle. The altitude proof is a useful invariant scaffold: put the origin at the intersection of two altitudes, translate perpendicularity into two dot-zero equations, and the third altitude follows by symmetry of the dot product.


In [ ]:
A = np.array([2.0, -0.35])
B = np.array([-0.55, 2.05])
rhs = np.array([np.dot(A, B), np.dot(A, B)])
C = np.linalg.solve(np.vstack([A, B]), rhs)
H = np.array([0.0, 0.0])
fig, ax = plt.subplots(figsize=(8.2, 6.1), facecolor="#fffdf8")
style_plane(ax, "altitudes become dot-zero constraints")
triangle_alt = np.vstack([A, B, C, A])
ax.plot(triangle_alt[:, 0], triangle_alt[:, 1], color=PALETTE["ink"], lw=2.2)
ax.fill(triangle_alt[:, 0], triangle_alt[:, 1], color=PALETTE["green"], alpha=0.07)
for P0 in [A, B, C]:
    ax.plot([P0[0], H[0]], [P0[1], H[1]], "--", color=PALETTE["red"], lw=1.8)
for label, point in {"u": A, "v": B, "w": C, "0 = orthocenter": H}.items():
    label_point(ax, label, point)
for P0, side_start, side_end in [(A, B, C), (B, A, C), (C, A, B)]:
    side = side_end - side_start
    foot = side_start + np.dot(P0 - side_start, side) / np.dot(side, side) * side
    ax.scatter([foot[0]], [foot[1]], s=35, color=PALETTE["red"], zorder=5)
ax.text(-0.82, -0.42, "u dot (w-v) = 0\nv dot (w-u) = 0\nso w dot (v-u) = 0", color=PALETTE["ink"], fontsize=10, bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="#d9dde2"))
ax.set_xlim(-0.95, 2.35)
ax.set_ylim(-0.75, 2.35)
inner_path = save_figure(fig, "inner-product-orthogonality-altitudes.png")
altitude_dot_residuals = {"u_dot_w_minus_v": float(np.dot(A, C - B)), "v_dot_w_minus_u": float(np.dot(B, C - A)), "w_dot_v_minus_u": float(np.dot(C, B - A))}
gram_alt = np.column_stack([A, B]).T @ np.column_stack([A, B])
inner_checks = {"altitude_dot_residuals": altitude_dot_residuals, "gram_matrix_for_u_v": gram_alt.tolist(), "gram_eigenvalues": np.linalg.eigvalsh(gram_alt).tolist(), "distance_squared_example": float(np.dot(B - A, B - A))}
CHECKS["inner_product"] = inner_checks
display_artifact(inner_path, width=760)
inner_checks


## 6. Cauchy-Schwarz and the triangle inequality

The Cauchy-Schwarz inequality says a dot product cannot be larger in absolute value than the product of the two lengths. In the plane, the exact reason is the Gram determinant identity: for `u=(a,b)` and `v=(c,d)`, the quantity `|u|^2 |v|^2 - (u dot v)^2` equals the square of the parallelogram area. Since a square is nonnegative, the inequality follows. The triangle inequality then says the direct vector `u+v` cannot be longer than walking first along `u` and then along `v`.


In [ ]:
u_cs = np.array([2.75, 0.95])
v_cs = np.array([1.15, 2.45])
proj_len = np.dot(u_cs, v_cs) / np.dot(v_cs, v_cs)
projection = proj_len * v_cs
slack_triangle = norm(u_cs) + norm(v_cs) - norm(u_cs + v_cs)
angles = np.linspace(0, math.pi, 37)
rows = []
for theta in angles:
    x = np.array([1.0, 0.0])
    y = np.array([math.cos(theta), math.sin(theta)])
    rows.append({"theta_radians": float(theta), "abs_dot_over_product": float(abs(np.dot(x, y)) / (norm(x) * norm(y))), "cauchy_schwarz_slack": float(norm(x) * norm(y) - abs(np.dot(x, y))), "triangle_slack_for_unit_pair": float(norm(x) + norm(y) - norm(x + y))})
angle_csv_path = save_csv(rows, "angle-sweep-checks.csv")

fig, axes = plt.subplots(1, 2, figsize=(12.6, 5.15), facecolor="#fffdf8")
ax = axes[0]
style_plane(ax, "Cauchy-Schwarz: projection length is bounded")
draw_vec(ax, [0, 0], v_cs, color="green", label="v")
draw_vec(ax, [0, 0], u_cs, color="blue", label="u")
ax.plot([u_cs[0], projection[0]], [u_cs[1], projection[1]], "--", color=PALETTE["red"], lw=1.8)
draw_vec(ax, [0, 0], projection, color="red", label="proj_v(u)", lw=2.0)
ax.scatter([projection[0]], [projection[1]], s=45, color=PALETTE["red"], zorder=5)
ax.text(0.35, 2.85, "|u dot v| <= |u||v|", color=PALETTE["ink"], fontsize=11, weight="bold")
ax.set_xlim(-0.25, 3.15)
ax.set_ylim(-0.25, 3.2)
ax = axes[1]
style_plane(ax, "triangle inequality: direct route is shortest")
draw_vec(ax, [0, 0], u_cs, color="blue", label="u")
draw_vec(ax, u_cs, v_cs, color="green", label="v after u")
draw_vec(ax, [0, 0], u_cs + v_cs, color="red", label="u+v")
ax.text(0.2, 3.65, f"|u| + |v| - |u+v| = {slack_triangle:.3f}", color=PALETTE["ink"], fontsize=11, weight="bold")
ax.set_xlim(-0.25, 4.35)
ax.set_ylim(-0.25, 3.95)
fig.suptitle("The Gram determinant controls both angle and shortest-path inequalities", fontsize=14, weight="bold", color=PALETTE["ink"])
inequality_path = save_figure(fig, "gram-cauchy-triangle-inequality.png")

a_sym, b_sym, c_sym, d_sym = sp.symbols("a b c d", real=True)
gram_gap = sp.expand((a_sym**2 + b_sym**2) * (c_sym**2 + d_sym**2) - (a_sym*c_sym + b_sym*d_sym)**2)
gram_factor = sp.factor(gram_gap)
assert gram_factor == (a_sym*d_sym - b_sym*c_sym) ** 2
inequality_checks = {"symbolic_gram_gap": str(gram_gap), "symbolic_gram_factor": str(gram_factor), "min_sample_cauchy_schwarz_slack": float(min(r["cauchy_schwarz_slack"] for r in rows)), "min_sample_triangle_slack": float(min(r["triangle_slack_for_unit_pair"] for r in rows)), "specific_triangle_slack": float(slack_triangle), "angle_sweep_rows": len(rows)}
CHECKS["cauchy_schwarz_triangle"] = inequality_checks
gram_check_path = save_check(inequality_checks, "gram-inequality-checks.json")
display_artifact(inequality_path, width=900)
inequality_checks


## 7. Rotations as matrices and complex multiplication

A rotation of the plane can be stored as the matrix `[[cos t, -sin t], [sin t, cos t]]`. The same operation can be stored as multiplication by the unit complex number `cos t + i sin t`. The two encodings agree point by point, and both make angle addition visible as composition.


In [ ]:
theta = 0.72
poly = np.array([[0.5, 0.25], [1.7, 0.45], [1.4, 1.15], [0.65, 1.35]])
R = rotation_matrix(theta)
rot_matrix_poly = (R @ poly.T).T
mult = complex(math.cos(theta), math.sin(theta))
rot_complex_poly = np.array([[ (mult * complex(point[0], point[1])).real, (mult * complex(point[0], point[1])).imag] for point in poly])
fig, axes = plt.subplots(1, 3, figsize=(14.6, 4.9), facecolor="#fffdf8")
ax = axes[0]
style_plane(ax, "rotated basis")
draw_vec(ax, [0, 0], [1, 0], color="gray", label="e1")
draw_vec(ax, [0, 0], [0, 1], color="gray", label="e2")
draw_vec(ax, [0, 0], R @ np.array([1.0, 0.0]), color="blue", label="R e1")
draw_vec(ax, [0, 0], R @ np.array([0.0, 1.0]), color="green", label="R e2")
ax.set_xlim(-1.25, 1.35)
ax.set_ylim(-1.1, 1.35)
ax = axes[1]
style_plane(ax, "same polygon from matrix and complex multiplication")
closed = np.vstack([poly, poly[0]])
closed_m = np.vstack([rot_matrix_poly, rot_matrix_poly[0]])
closed_c = np.vstack([rot_complex_poly, rot_complex_poly[0]])
ax.plot(closed[:, 0], closed[:, 1], color=PALETTE["gray"], lw=2, label="original")
ax.plot(closed_m[:, 0], closed_m[:, 1], color=PALETTE["blue"], lw=2.4, label="matrix")
ax.plot(closed_c[:, 0], closed_c[:, 1], "--", color=PALETTE["orange"], lw=2.0, label="complex")
ax.legend(loc="upper left", fontsize=9)
ax.set_xlim(-1.0, 2.0)
ax.set_ylim(-0.1, 2.1)
ax = axes[2]
style_plane(ax, "composition: R(a)R(b) = R(a+b)")
angle_pairs = [(0.2, 0.55), (0.8, -0.35), (1.1, 0.45), (-0.4, 1.2)]
residuals = []
for idx, (t1, t2) in enumerate(angle_pairs):
    point = np.array([1.0 + 0.18 * idx, 0.25 + 0.11 * idx])
    first_then_second = rotation_matrix(t1) @ (rotation_matrix(t2) @ point)
    together = rotation_matrix(t1 + t2) @ point
    residuals.append(norm(first_then_second - together))
    ax.plot([first_then_second[0], together[0]], [first_then_second[1], together[1]], color=PALETTE["red"], lw=2)
    ax.scatter([together[0]], [together[1]], s=45, color=PALETTE["purple"])
ax.text(-0.8, 1.25, f"max composition\nresidual = {max(residuals):.1e}", color=PALETTE["ink"], fontsize=10, bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="#d9dde2"))
ax.set_xlim(-1.2, 1.65)
ax.set_ylim(-0.4, 1.75)
fig.suptitle("A plane rotation has two equivalent encodings", fontsize=14, weight="bold", color=PALETTE["ink"])
rotation_path = save_figure(fig, "rotation-matrix-complex-action.png")

z_points = np.array([0.2 + 0.2j, 1.4 + 0.1j, 1.15 + 0.95j, 0.25 + 1.15j, 0.2 + 0.2j])
angles = np.linspace(0, 2 * math.pi, 36)
frames = []
for t0 in angles:
    m0 = complex(math.cos(t0), math.sin(t0))
    complex_rot = m0 * z_points
    matrix_rot = (rotation_matrix(t0) @ np.column_stack([z_points.real, z_points.imag]).T).T
    frames.append(go.Frame(data=[go.Scatter(x=z_points.real, y=z_points.imag, mode="lines+markers", name="original", line=dict(color="#78818c")), go.Scatter(x=matrix_rot[:, 0], y=matrix_rot[:, 1], mode="lines+markers", name="matrix rotation", line=dict(color="#1f77b4")), go.Scatter(x=complex_rot.real, y=complex_rot.imag, mode="lines", name="complex rotation", line=dict(color="#e07a2f", dash="dash"))], name=f"{t0:.3f}"))
fig_html = go.Figure(data=frames[0].data, frames=frames)
fig_html.update_layout(title="Rotation slider: matrix path and complex path stay locked together", xaxis=dict(scaleanchor="y", scaleratio=1, range=[-1.8, 1.8], zeroline=True), yaxis=dict(range=[-1.8, 1.8], zeroline=True), width=760, height=620, updatemenus=[{"type": "buttons", "buttons": [{"label": "play", "method": "animate", "args": [None, {"frame": {"duration": 80, "redraw": True}, "fromcurrent": True}]}]}], sliders=[{"steps": [{"method": "animate", "label": f"{t0/math.pi:.2f} pi", "args": [[f"{t0:.3f}"], {"mode": "immediate", "frame": {"duration": 0, "redraw": True}}]} for t0 in angles]}])
rotation_html_path = save_html_figure(fig_html, "rotation-composition-lab.html")
x_sym, y_sym, t_sym = sp.symbols("x y t", real=True)
rot_sym = sp.Matrix([[sp.cos(t_sym), -sp.sin(t_sym)], [sp.sin(t_sym), sp.cos(t_sym)]])
rotation_checks = {"matrix_complex_polygon_max_error": float(np.max(np.linalg.norm(rot_matrix_poly - rot_complex_poly, axis=1))), "orthogonality_residual": float(np.max(np.abs(R.T @ R - np.eye(2)))), "determinant": float(np.linalg.det(R)), "composition_residuals": residuals, "symbolic_RtR_minus_I": str(sp.simplify(rot_sym.T * rot_sym - sp.eye(2)))}
CHECKS["rotations"] = rotation_checks
rotation_check_path = save_check(rotation_checks, "rotation-checks.json")
display_artifact(rotation_path, width=980)
display_artifact(rotation_html_path, width="100%", height=620)
rotation_checks


## 8. Applied lab: a vector navigation traverse

A small survey traverse makes the chapter's objects work together. Each leg is a vector obtained from a length and a heading. Headings are updated by multiplying the current unit complex direction by `cos turn + i sin turn`. The path endpoint is the sum of the leg vectors, the point cloud centroid is the average of the visited positions, and the Gram matrix of the step vectors records their inner-product geometry.


In [ ]:
lengths = np.array([1.55, 1.05, 1.35, 0.9, 1.2, 0.85])
turns = np.deg2rad([32, -46, 58, 22, -74, 38])
heading = complex(1.0, 0.0)
position = complex(0.0, 0.0)
positions = [position]
step_vectors, lab_rows = [], []
heading_angle = 0.0
for idx, (length, turn) in enumerate(zip(lengths, turns), start=1):
    heading *= complex(math.cos(turn), math.sin(turn))
    heading_angle += turn
    step = length * heading
    position += step
    positions.append(position)
    step_vectors.append(np.array([step.real, step.imag]))
    lab_rows.append({"leg": idx, "length": float(length), "turn_degrees": float(math.degrees(turn)), "heading_degrees": float(math.degrees(heading_angle)), "dx": float(step.real), "dy": float(step.imag), "x": float(position.real), "y": float(position.imag)})
positions_arr = np.array([[z.real, z.imag] for z in positions])
steps_arr = np.vstack(step_vectors)
endpoint = positions_arr[-1]
centroid_lab = positions_arr.mean(axis=0)
path_length = float(lengths.sum())
direct_distance = norm(endpoint)
gram_steps = steps_arr @ steps_arr.T
gram_eigs = np.linalg.eigvalsh(gram_steps)
nav_csv_path = save_csv(lab_rows, "navigation-steps.csv")

fig, axes = plt.subplots(1, 2, figsize=(12.8, 5.45), facecolor="#fffdf8")
ax = axes[0]
style_plane(ax, "survey traverse: endpoint is the vector sum")
ax.plot(positions_arr[:, 0], positions_arr[:, 1], "-o", color=PALETTE["blue"], lw=2.2, markersize=5)
for i, step in enumerate(steps_arr):
    draw_vec(ax, positions_arr[i], step, color="blue", lw=1.7, alpha=0.55)
ax.scatter([centroid_lab[0]], [centroid_lab[1]], s=115, color=PALETTE["gold"], edgecolor=PALETTE["ink"], zorder=5)
ax.text(centroid_lab[0] + 0.05, centroid_lab[1] + 0.06, "centroid", color=PALETTE["ink"], fontsize=10, weight="bold")
draw_vec(ax, [0, 0], endpoint, color="red", label="sum")
ax.text(endpoint[0] + 0.04, endpoint[1] + 0.04, "endpoint", color=PALETTE["red"], fontsize=10, weight="bold")
ax.set_xlim(-0.3, max(positions_arr[:, 0]) + 0.75)
ax.set_ylim(min(positions_arr[:, 1]) - 0.45, max(positions_arr[:, 1]) + 0.55)
ax = axes[1]
ax.set_title("inner-product diagnostics for the legs", color=PALETTE["ink"], fontsize=12, weight="bold", pad=10)
im = ax.imshow(gram_steps, cmap="viridis")
ax.set_xticks(range(len(lengths)))
ax.set_yticks(range(len(lengths)))
ax.set_xticklabels([f"s{i}" for i in range(1, len(lengths) + 1)])
ax.set_yticklabels([f"s{i}" for i in range(1, len(lengths) + 1)])
for i in range(len(lengths)):
    for j in range(len(lengths)):
        ax.text(j, i, f"{gram_steps[i, j]:.2f}", ha="center", va="center", color="white" if gram_steps[i, j] > 0.4 else PALETTE["ink"], fontsize=8)
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="dot product")
ax.text(-0.4, len(lengths) + 0.05, f"path length {path_length:.2f}; direct distance {direct_distance:.2f}", color=PALETTE["ink"], fontsize=10)
fig.suptitle("Applied lab: vector addition, centroids, inner products, and complex rotations", fontsize=14, weight="bold", color=PALETTE["ink"])
nav_path = save_figure(fig, "applied-navigation-vector-lab.png")

fig_nav = go.Figure()
fig_nav.add_trace(go.Scatter(x=positions_arr[:, 0], y=positions_arr[:, 1], mode="lines+markers+text", text=[str(i) for i in range(len(positions_arr))], textposition="top center", name="visited positions"))
fig_nav.add_trace(go.Scatter(x=[0, endpoint[0]], y=[0, endpoint[1]], mode="lines+markers", name="direct displacement", line=dict(color="#c74343", width=4)))
fig_nav.add_trace(go.Scatter(x=[centroid_lab[0]], y=[centroid_lab[1]], mode="markers+text", text=["centroid"], textposition="bottom right", marker=dict(size=12, color="#c7971d"), name="centroid"))
fig_nav.update_layout(title="Interactive traverse: compare path, direct displacement, and centroid", xaxis=dict(scaleanchor="y", scaleratio=1, zeroline=True), yaxis=dict(zeroline=True), width=760, height=560)
nav_html_path = save_html_figure(fig_nav, "vector-navigation-lab.html")
nav_checks = {"endpoint_from_accumulated_positions": endpoint.tolist(), "endpoint_from_step_sum": steps_arr.sum(axis=0).tolist(), "endpoint_sum_residual": norm(endpoint - steps_arr.sum(axis=0)), "centroid": centroid_lab.tolist(), "path_length": path_length, "direct_distance": direct_distance, "triangle_inequality_slack_path_minus_direct": path_length - direct_distance, "gram_min_eigenvalue": float(gram_eigs.min()), "gram_symmetry_residual": float(np.max(np.abs(gram_steps - gram_steps.T)))}
CHECKS["applied_navigation_lab"] = nav_checks
nav_check_path = save_check(nav_checks, "applied-lab-checks.json")
display_artifact(nav_path, width=930)
display_artifact(nav_html_path, width="100%", height=560)
nav_checks


## Reading the artifacts as one argument

The artifacts are meant to be read in sequence, not as isolated pictures. Start with the parallelogram and dilation figure and ignore length completely: the only promises being checked are closure of addition, repeated travel, and signed scaling from the origin. Then move to the direction figure and watch how a determinant changes the language. A nonzero determinant says that two direction lines give independent coordinates; a zero determinant says a proposed parallel relation has no area between its direction vectors.

The centroid figures deliberately keep the same averaging idea in two dimensions and three dimensions. The triangle median point is not a new construction rule; it is the average already suggested by midpoints. The tetrahedron HTML extends the same coefficient pattern, changing the visible fraction from two-thirds to three-quarters because the opposite face average already contains three vertices.

The inner-product and inequality figures add measurement only after the vector-space operations are in place. Dot-zero altitudes show how perpendicularity becomes an equation. The Gram determinant check then turns a visual area into an algebraic square, which is why Cauchy-Schwarz is not a numerical accident. Finally, the rotation and navigation labs show that matrices, complex numbers, sums, centroids, and Gram matrices are compatible tools for the same Euclidean plane.


## Final sanity checks

The final cell checks the mathematical claims and the files the notebook generated. It also asserts that the old generic scaffold filenames are no longer present in this chapter's artifact subtree.


In [ ]:
manifest = {category: [path.relative_to(ARTIFACT_ROOT).as_posix() for path in paths] for category, paths in ARTIFACTS.items()}
manifest_path = save_check(manifest, "artifact-manifest.json")
artifact_paths = [path for paths in ARTIFACTS.values() for path in paths]
assert_artifacts(artifact_paths, min_size=256)
assert max(vector_residuals.values()) < EPS
assert direction_checks["rank_p_q"] == 2
assert abs(direction_checks["thales_parallel_residual"]) < EPS
assert proof_checks["is_dag"]
assert max(centroid_checks["triangle_median_residuals"].values()) < EPS
assert max(centroid_checks["tetrahedron_vertex_to_face_residuals"]) < EPS
assert max(abs(v) for v in altitude_dot_residuals.values()) < EPS
assert inequality_checks["symbolic_gram_factor"] == "(a*d - b*c)**2"
assert inequality_checks["min_sample_cauchy_schwarz_slack"] >= -1e-12
assert inequality_checks["min_sample_triangle_slack"] >= -1e-12
assert rotation_checks["matrix_complex_polygon_max_error"] < EPS
assert rotation_checks["orthogonality_residual"] < EPS
assert abs(rotation_checks["determinant"] - 1.0) < EPS
assert max(rotation_checks["composition_residuals"]) < EPS
assert nav_checks["endpoint_sum_residual"] < EPS
assert nav_checks["triangle_inequality_slack_path_minus_direct"] >= -EPS
assert nav_checks["gram_min_eigenvalue"] >= -1e-9
for rel in STALE_GENERIC_ARTIFACTS:
    assert not (ARTIFACT_ROOT / rel).exists(), f"stale generic artifact remains: {rel}"
png_stats = [image_stats(path) for path in ARTIFACTS["figures"]]
for item in png_stats:
    assert item["width"] >= 300 and item["height"] >= 240
    assert item["pixel_std"] > 2.0
final_sanity = {"source_span": SOURCE_SPAN, "figure_count": len(ARTIFACTS["figures"]), "html_count": len(ARTIFACTS["html"]), "check_count": len(ARTIFACTS["checks"]) + 1, "table_count": len(ARTIFACTS["tables"]), "max_vector_law_residual": max(vector_residuals.values()), "max_altitude_dot_residual": max(abs(v) for v in altitude_dot_residuals.values()), "rotation_matrix_complex_max_error": rotation_checks["matrix_complex_polygon_max_error"], "navigation_path_slack": nav_checks["triangle_inequality_slack_path_minus_direct"], "png_min_pixel_std": min(item["pixel_std"] for item in png_stats)}
final_sanity_path = save_check(final_sanity, "final-sanity.json")
manifest = {category: [path.relative_to(ARTIFACT_ROOT).as_posix() for path in paths] for category, paths in ARTIFACTS.items()}
save_json(manifest, ARTIFACT_ROOT, "checks", "artifact-manifest.json")
display(final_sanity)


## Takeaways

- Vector addition and scalar multiplication explain parallel transfer, midpoint formulas, and centroids before length enters the story.
- Linear independence is the condition that lets coefficient comparisons mean something geometrically; in the plane it is the nonzero area of a determinant.
- The inner product is the bridge from vector algebra to Euclidean measurement: length, perpendicularity, angle, and the cosine rule all come from it.
- Cauchy-Schwarz is a Gram determinant statement, and the triangle inequality is its immediate geometric consequence.
- Plane rotations can be encoded either by a 2 by 2 matrix or by multiplying complex numbers on the unit circle; the equality of these two encodings is visible point by point and testable by residuals.
